# 01 - Flat 34-Class XGBoost Ablation

Purpose: train a fair single-stage 34-class XGBoost baseline using the locked processed-data protocol.

Quality rule: run from a clean kernel, save all artifacts, and do not use the test set for training or tuning.


## 1. Imports

Load only the libraries needed for the flat XGBoost baseline, tuning, timing, and artifact saving.


In [12]:
from __future__ import annotations

import sys
import time
from pathlib import Path

import joblib
import numpy as np
import optuna
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier


## 2. Local Helper Imports

Resolve the Ablation Study folder robustly and import the shared helper functions used by all ablation notebooks.


In [13]:
ABLATION_DIR = Path.cwd()
if not (ABLATION_DIR / 'ablation_common.py').exists():
    ABLATION_DIR = Path(r'D:/Ml Project/prepared_final-20260304T163925Z-3-001/prepared_final/ML Project_ FINAL/Hierarchical IoT Intrusion Detection/Hierarchical-IoT-Intrusion-Detection/Ablation Study')
sys.path.insert(0, str(ABLATION_DIR))

from ablation_common import (
    evaluate_34class,
    ensure_dir,
    load_base_feature_names,
    load_label_encoder_34,
    project_root,
    save_npz_compressed,
    workspace_root,
    write_json,
)


## 3. Production Configuration

Define the full-run training settings and canonical output paths. This notebook uses only the internal processed train/test splits.


In [14]:
RUN_OPTUNA = True
N_TRIALS = 40
OPTUNA_SHOW_PROGRESS = False
RANDOM_STATE = 42

REQUIRE_XGBOOST_GPU = True
XGB_DEVICE = 'cuda'

TUNING_FIT_MAX_ROWS = 600_000
TUNING_VALID_MAX_ROWS = 200_000
CALIBRATION_SIZE = 0.15

WORKSPACE_ROOT = workspace_root()
PROJECT_ROOT = project_root()
PROCESSED_DIR = WORKSPACE_ROOT / 'processed_data'
RESULTS_DIR = ensure_dir(PROJECT_ROOT / 'results' / 'ablation' / 'flat')
MODELS_DIR = ensure_dir(PROJECT_ROOT / 'models' / 'ablation')

MODEL_PATH = MODELS_DIR / 'flat_34class_xgboost_optuna40.joblib'
PRED_PATH = RESULTS_DIR / 'flat_34class_test_predictions.npz'
METRICS_PATH = RESULTS_DIR / 'flat_34class_test_metrics.json'
REPORT_PATH = RESULTS_DIR / 'flat_34class_per_class_report.csv'
PARAMS_PATH = RESULTS_DIR / 'flat_34class_best_params.json'


## 4. XGBoost GPU Validation

Run a tiny real fit on CUDA and fail early if XGBoost silently falls back to CPU.


In [15]:
def validate_xgboost_gpu():
    X_probe = np.random.default_rng(RANDOM_STATE).random((512, 8), dtype=np.float32)
    y_probe = np.random.default_rng(RANDOM_STATE + 1).integers(0, 3, size=512, dtype=np.int64)
    probe = XGBClassifier(
        objective='multi:softprob',
        num_class=3,
        eval_metric='mlogloss',
        tree_method='hist',
        device=XGB_DEVICE,
        n_estimators=2,
        max_depth=2,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=0,
    )
    try:
        probe.fit(X_probe, y_probe, verbose=False)
    except Exception as exc:
        if REQUIRE_XGBOOST_GPU:
            raise RuntimeError('XGBoost CUDA probe failed; do not continue on CPU.') from exc
        return 'cpu'

    booster_config = probe.get_booster().save_config()
    uses_cuda = '"device":"cuda' in booster_config
    uses_gpu_hist = 'grow_gpu_hist' in booster_config
    if REQUIRE_XGBOOST_GPU and not (uses_cuda and uses_gpu_hist):
        raise RuntimeError('XGBoost did not use CUDA/grow_gpu_hist in the probe fit.')
    return XGB_DEVICE if uses_cuda else 'cpu'

ACTIVE_XGB_DEVICE = validate_xgboost_gpu()
print('XGBoost training device:', ACTIVE_XGB_DEVICE)


XGBoost training device: cuda


## 5. Required Artifact Check

Fail early if the locked processed train/test files or label/feature encoders are missing.


In [16]:
required_files = [
    PROCESSED_DIR / 'train_data.npz',
    PROCESSED_DIR / 'test_data.npz',
    PROCESSED_DIR / 'label_encoder_34.joblib',
    PROCESSED_DIR / 'feature_columns.joblib',
]
missing = [str(path) for path in required_files if not path.exists()]
assert not missing, missing
print('All required processed-data artifacts exist.')


All required processed-data artifacts exist.


## 6. Load Locked Internal Data

Load the 46-feature processed train/test split and verify that all 34 classes are present in both splits.


In [17]:
with np.load(PROCESSED_DIR / 'train_data.npz') as train_npz:
    X_train_full = train_npz['X']
    y_train_full = train_npz['y_34class'].astype(np.int64)

with np.load(PROCESSED_DIR / 'test_data.npz') as test_npz:
    X_test = test_npz['X']
    y_test = test_npz['y_34class'].astype(np.int64)

le34 = load_label_encoder_34()
label_names = list(le34.classes_)
feature_names = load_base_feature_names()
expected_classes = np.arange(34, dtype=np.int64)

assert len(feature_names) == 46
assert len(label_names) == 34
assert X_train_full.shape[1] == len(feature_names)
assert X_test.shape[1] == len(feature_names)
assert np.array_equal(np.unique(y_train_full), expected_classes)
assert np.array_equal(np.unique(y_test), expected_classes)
print('Train:', X_train_full.shape, 'Test:', X_test.shape)


Train: (5599994, 46) Test: (1399999, 46)


## 7. Stratified Tuning Subsample Helper

Keep Optuna tuning computationally feasible while preserving every class in a balanced internal sample.


In [18]:
def stratified_limit_indices(y, max_rows=None, rows_per_class=None, random_state=42):
    rng = np.random.default_rng(random_state)
    selected = []
    classes = np.unique(y)
    if rows_per_class is not None:
        for cls in classes:
            idx = np.where(y == cls)[0]
            take = min(rows_per_class, len(idx))
            selected.append(rng.choice(idx, size=take, replace=False))
    elif max_rows is not None and len(y) > max_rows:
        per_class = max(1, max_rows // len(classes))
        for cls in classes:
            idx = np.where(y == cls)[0]
            take = min(per_class, len(idx))
            selected.append(rng.choice(idx, size=take, replace=False))
    else:
        return np.arange(len(y))
    out = np.concatenate(selected)
    rng.shuffle(out)
    return out


## 8. Internal Calibration Split

Create a stratified train/calibration split from training data only; the internal test split is not touched here.


In [19]:
X_train_work = X_train_full
y_train_work = y_train_full
X_test_eval = X_test
y_test_eval = y_test
test_row_id = np.arange(len(y_test), dtype=np.int64)

X_fit, X_calib, y_fit, y_calib = train_test_split(
    X_train_work,
    y_train_work,
    test_size=CALIBRATION_SIZE,
    stratify=y_train_work,
    random_state=RANDOM_STATE,
)
assert np.array_equal(np.unique(y_fit), expected_classes)
assert np.array_equal(np.unique(y_calib), expected_classes)
print('Fit:', X_fit.shape, 'Calibration:', X_calib.shape, 'Eval:', X_test_eval.shape)


Fit: (4759994, 46) Calibration: (840000, 46) Eval: (1399999, 46)


## 9. Tuning Data and Weights

Build balanced tuning subsets and class-balanced sample weights so rare classes are not ignored.


In [20]:
tune_fit_idx = stratified_limit_indices(y_fit, max_rows=TUNING_FIT_MAX_ROWS, random_state=RANDOM_STATE)
tune_calib_idx = stratified_limit_indices(y_calib, max_rows=TUNING_VALID_MAX_ROWS, random_state=RANDOM_STATE + 1)
X_tune_fit = X_fit[tune_fit_idx]
y_tune_fit = y_fit[tune_fit_idx]
X_tune_valid = X_calib[tune_calib_idx]
y_tune_valid = y_calib[tune_calib_idx]

assert np.array_equal(np.unique(y_tune_fit), expected_classes)
assert np.array_equal(np.unique(y_tune_valid), expected_classes)
sample_weight_tune = compute_sample_weight(class_weight='balanced', y=y_tune_fit)
print('Tuning fit:', X_tune_fit.shape, 'Tuning valid:', X_tune_valid.shape)


Tuning fit: (484431, 46) Tuning valid: (146961, 46)


## 10. XGBoost Search Space

Define the default flat baseline parameters and the Optuna search ranges used for the 34-class model.


In [21]:
def make_xgb_params(trial=None):
    if trial is None:
        return {
            'n_estimators': 700,
            'max_depth': 8,
            'learning_rate': 0.05,
            'subsample': 0.9,
            'colsample_bytree': 0.9,
            'min_child_weight': 3,
            'gamma': 0.0,
            'reg_alpha': 0.0,
            'reg_lambda': 1.0,
        }
    return {
        'n_estimators': trial.suggest_int('n_estimators', 300, 1000),
        'max_depth': trial.suggest_int('max_depth', 4, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.65, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.65, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0.0, 3.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 2.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.5, 5.0),
    }


## 11. Model Factory and Objective

Build a CPU histogram XGBoost classifier and optimize macro-F1 on the internal calibration subset.


In [22]:
def build_model(params):
    return XGBClassifier(
        objective='multi:softprob',
        num_class=34,
        eval_metric='mlogloss',
        tree_method='hist',
        device=ACTIVE_XGB_DEVICE,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=0,
        **params,
    )

def objective(trial):
    params = make_xgb_params(trial)
    model = build_model(params)
    model.fit(X_tune_fit, y_tune_fit, sample_weight=sample_weight_tune, verbose=False)
    pred = model.predict(X_tune_valid)
    return f1_score(y_tune_valid, pred, average='macro', zero_division=0)


## 12. Run Hyperparameter Search

Run the configured Optuna study and save the selected parameters with reproducibility metadata.


In [23]:
if RUN_OPTUNA:
    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=OPTUNA_SHOW_PROGRESS)
    best_params = make_xgb_params(None)
    best_params.update(study.best_params)
    best_score = float(study.best_value)
else:
    best_params = make_xgb_params(None)
    best_score = None

write_json(
    PARAMS_PATH,
    {
        'best_params': best_params,
        'best_validation_macro_f1': best_score,
        'run_optuna': bool(RUN_OPTUNA),
        'n_trials': int(N_TRIALS) if RUN_OPTUNA else 0,
        'tuning_fit_max_rows': int(TUNING_FIT_MAX_ROWS),
        'tuning_valid_max_rows': int(TUNING_VALID_MAX_ROWS),
        'calibration_size': float(CALIBRATION_SIZE),
        'random_state': int(RANDOM_STATE),
        'xgb_device': ACTIVE_XGB_DEVICE,
        'require_xgboost_gpu': bool(REQUIRE_XGBOOST_GPU),
        'feature_count': int(len(feature_names)),
        'class_count': int(len(label_names)),
    },
)
best_params


[I 2026-06-07 12:44:52,607] A new study created in memory with name: no-name-ae2d321e-f563-4054-adbd-749bc0c26ae1
[I 2026-06-07 12:45:47,192] Trial 0 finished with value: 0.8838379163810555 and parameters: {'n_estimators': 562, 'max_depth': 12, 'learning_rate': 0.08960785365368121, 'subsample': 0.8595304694689628, 'colsample_bytree': 0.7046065241548528, 'min_child_weight': 2, 'gamma': 0.17425083650459838, 'reg_alpha': 1.7323522915498704, 'reg_lambda': 3.20501755284444}. Best is trial 0 with value: 0.8838379163810555.
[I 2026-06-07 12:46:16,491] Trial 1 finished with value: 0.8603137640461467 and parameters: {'n_estimators': 796, 'max_depth': 4, 'learning_rate': 0.18276027831785724, 'subsample': 0.9413549242801476, 'colsample_bytree': 0.7243186887373967, 'min_child_weight': 2, 'gamma': 0.5502135295603015, 'reg_alpha': 0.6084844859190754, 'reg_lambda': 2.86140394234507}. Best is trial 0 with value: 0.8838379163810555.
[I 2026-06-07 12:46:47,530] Trial 2 finished with value: 0.85815449814

{'n_estimators': 902,
 'max_depth': 7,
 'learning_rate': 0.13032559568934732,
 'subsample': 0.8570273234637671,
 'colsample_bytree': 0.7744904147775846,
 'min_child_weight': 1,
 'gamma': 0.011503558388925588,
 'reg_alpha': 0.38292114098568447,
 'reg_lambda': 3.5921990376933244}

## 13. Train Final Flat Model

Train the selected flat 34-class XGBoost model on the full processed training split and save it.


In [24]:
sample_weight_final = compute_sample_weight(class_weight='balanced', y=y_train_work)
flat_model = build_model(best_params)
start = time.perf_counter()
flat_model.fit(X_train_work, y_train_work, sample_weight=sample_weight_final, verbose=False)
training_seconds = time.perf_counter() - start

assert hasattr(flat_model, 'classes_')
assert np.array_equal(flat_model.classes_.astype(np.int64), expected_classes)
final_booster_config = flat_model.get_booster().save_config()
assert '"device":"cuda' in final_booster_config
assert 'grow_gpu_hist' in final_booster_config
joblib.dump(flat_model, MODEL_PATH)
print('Saved model:', MODEL_PATH)
print('Training seconds:', training_seconds)


Saved model: D:\Ml Project\prepared_final-20260304T163925Z-3-001\prepared_final\ML Project_ FINAL\Hierarchical IoT Intrusion Detection\Hierarchical-IoT-Intrusion-Detection\models\ablation\flat_34class_xgboost_optuna40.joblib
Training seconds: 410.6052405000082


## 14. Evaluate Internal Test Split

Evaluate on the locked processed internal test split for diagnostics, not for hyperparameter or threshold selection.


In [25]:
start = time.perf_counter()
y_proba = flat_model.predict_proba(X_test_eval)
y_pred = np.argmax(y_proba, axis=1).astype(np.int64)
inference_seconds = time.perf_counter() - start

assert y_proba.shape == (len(y_test_eval), len(label_names))
metrics, per_class = evaluate_34class(
    y_test_eval, y_pred, y_proba, label_names, inference_seconds, 'Flat 34-class XGBoost'
)
metrics['training_seconds'] = float(training_seconds)
metrics['training_rows'] = int(len(y_train_work))
metrics['internal_test_rows'] = int(len(y_test_eval))
metrics['feature_count'] = int(len(feature_names))
metrics['class_count'] = int(len(label_names))
metrics['n_trials'] = int(N_TRIALS) if RUN_OPTUNA else 0
metrics['xgb_device'] = ACTIVE_XGB_DEVICE
write_json(METRICS_PATH, metrics)
per_class.to_csv(REPORT_PATH, index=False)
metrics


{'model': 'Flat 34-class XGBoost',
 'rows': 1399999,
 'accuracy': 0.9722392658851899,
 'balanced_accuracy': 0.8860524981740066,
 'macro_precision': 0.8704995849404229,
 'macro_recall': 0.8860524981740066,
 'macro_f1': 0.876287767384643,
 'weighted_f1': 0.9733090925691309,
 'mcc': 0.9701235671532547,
 'elapsed_seconds': 5.094333500019275,
 'us_per_flow': 3.638812242022512,
 'mean_top1_confidence': 0.9662755131721497,
 'training_seconds': 410.6052405000082,
 'training_rows': 5599994,
 'internal_test_rows': 1399999,
 'feature_count': 46,
 'class_count': 34,
 'n_trials': 40,
 'xgb_device': 'cuda'}

## 15. Save Internal Predictions

Persist predictions, probabilities, labels, and feature metadata for audit and downstream diagnostics.


In [26]:
save_npz_compressed(
    PRED_PATH,
    y_true_34class=y_test_eval.astype(np.int64),
    y_pred_34class=y_pred.astype(np.int64),
    y_proba_34class=y_proba.astype(np.float32),
    test_row_id=test_row_id.astype(np.int64),
    label_names=np.array(label_names, dtype=object),
    feature_names=np.array(feature_names, dtype=object),
    xgb_device=np.array([ACTIVE_XGB_DEVICE], dtype=object),
    source_protocol=np.array(['processed_data/test_data.npz'], dtype=object),
)
print('Saved predictions:', PRED_PATH)
print('Saved metrics:', METRICS_PATH)
print('Saved report:', REPORT_PATH)


Saved predictions: D:\Ml Project\prepared_final-20260304T163925Z-3-001\prepared_final\ML Project_ FINAL\Hierarchical IoT Intrusion Detection\Hierarchical-IoT-Intrusion-Detection\results\ablation\flat\flat_34class_test_predictions.npz
Saved metrics: D:\Ml Project\prepared_final-20260304T163925Z-3-001\prepared_final\ML Project_ FINAL\Hierarchical IoT Intrusion Detection\Hierarchical-IoT-Intrusion-Detection\results\ablation\flat\flat_34class_test_metrics.json
Saved report: D:\Ml Project\prepared_final-20260304T163925Z-3-001\prepared_final\ML Project_ FINAL\Hierarchical IoT Intrusion Detection\Hierarchical-IoT-Intrusion-Detection\results\ablation\flat\flat_34class_per_class_report.csv


## 16. Artifact Contract Audit

Validate saved outputs so later notebooks can safely load the model and prediction artifacts.


In [27]:
assert MODEL_PATH.exists()
assert PRED_PATH.exists()
assert METRICS_PATH.exists()
loaded = np.load(PRED_PATH, allow_pickle=True)
assert loaded['y_true_34class'].shape == loaded['y_pred_34class'].shape
assert loaded['y_proba_34class'].shape == (len(loaded['y_true_34class']), 34)
assert len(loaded['test_row_id']) == len(loaded['y_true_34class'])
assert len(loaded['feature_names']) == 46
assert len(loaded['label_names']) == 34
assert loaded['xgb_device'][0] == 'cuda'
assert np.array_equal(np.unique(loaded['y_true_34class']), expected_classes)
print('Notebook 01 artifact audit passed.')


Notebook 01 artifact audit passed.


In [29]:
import pandas as pd
# Flat internal metrics summary
flat_summary = pd.DataFrame([{
    'model': metrics['model'],
    'rows': metrics['rows'],
    'accuracy': metrics['accuracy'],
    'macro_f1': metrics['macro_f1'],
    'weighted_f1': metrics['weighted_f1'],
    'macro_recall': metrics['macro_recall'],
    'mcc': metrics['mcc'],
    'us_per_flow': metrics['us_per_flow'],
    'training_seconds': metrics['training_seconds'],
    'xgb_device': metrics.get('xgb_device', 'unknown'),
}])
flat_summary

,model,rows,accuracy,macro_f1,weighted_f1,macro_recall,mcc,us_per_flow,training_seconds,xgb_device
0,Flat 34-class XGBoost,1399999,0.972239,0.876288,0.973309,0.886052,0.970124,3.638812,410.605241,cuda


In [30]:
# Worst-performing classes by F1
flat_per_class_only = per_class[per_class['label'].isin(label_names)].copy()
worst_10_flat_classes = (
    flat_per_class_only
    .sort_values(['f1-score', 'support'], ascending=[True, True])
    .head(10)
)
worst_10_flat_classes

,label,precision,recall,f1-score,support
33,XSS,0.538198,0.597468,0.566287,790.0
28,Recon-PingSweep,0.559091,0.597087,0.577465,412.0
17,DictionaryBruteForce,0.502675,0.755071,0.603548,2613.0
0,Backdoor_Malware,0.651943,0.577465,0.612448,639.0
31,Uploading_Attack,0.750000,0.530973,0.621762,226.0
3,CommandInjection,0.617516,0.645794,0.631339,1070.0
30,SqlInjection,0.680633,0.670027,0.675289,1091.0
2,BrowserHijacking,0.647059,0.707826,0.676080,1150.0
27,Recon-OSScan,0.631074,0.759562,0.689382,19818.0
29,Recon-PortScan,0.663013,0.795966,0.723432,16262.0


In [33]:
FIGURES_DIR = PROJECT_ROOT / 'figures' / 'ablation'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [38]:

import matplotlib.pyplot as plt

FIGURES_DIR = PROJECT_ROOT / 'figures' / 'ablation'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

flat_per_class_only = per_class[per_class['label'].isin(label_names)].copy()
plot_df = flat_per_class_only.sort_values('f1-score', ascending=True)

ax = plot_df.set_index('label')['f1-score'].plot(
    kind='barh',
    figsize=(10, 9),
    color='#2563eb'
)
ax.set_title('Flat XGBoost Internal Test Per-Class F1')
ax.set_xlim(0, 1.05)
ax.set_xlabel('F1-score')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plot_path = FIGURES_DIR / 'flat_internal_per_class_f1.png'
plt.savefig(plot_path, dpi=200)
plt.close()

print('Saved:', plot_path)

Saved: D:\Ml Project\prepared_final-20260304T163925Z-3-001\prepared_final\ML Project_ FINAL\Hierarchical IoT Intrusion Detection\Hierarchical-IoT-Intrusion-Detection\figures\ablation\flat_internal_per_class_f1.png
